In [1]:
import pandas as pd
import json
import os

import torch
import torch.nn.functional as F
from torchcodec.decoders import AudioDecoder
from torchinfo import summary

from vox_profile_release.src.model.fluency.whisper_fluency import WhisperWrapper
from utils import slice_range

In [2]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
device

'cuda:0'

In [3]:
SAMPLE_RATE = 16000
CHANNELS = 1
WINDOW = 2
STRIDE = 0.5
DISFLUENCY_THRESHOLD = 0.9

In [4]:
rec_path = os.path.join('inputs', 'rec_pre.mp3')
out_path = os.path.join('outputs', 'segments_fluency')

In [5]:
model = WhisperWrapper.from_pretrained('tiantiaf/whisper-large-v3-speech-flow').to(device)

Some weights of WhisperModel were not initialized from the model checkpoint at openai/whisper-large-v3 and are newly initialized because the shapes did not match:
- model.encoder.embed_positions.weight: found shape torch.Size([1500, 1280]) in the checkpoint and torch.Size([150, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
summary(model)

Layer (type:depth-idx)                                  Param #
WhisperWrapper                                          192,033
├─WhisperModel: 1-1                                     --
│    └─WhisperEncoder: 2-1                              --
│    │    └─Conv1d: 3-1                                 (492,800)
│    │    └─Conv1d: 3-2                                 (4,916,480)
│    │    └─Embedding: 3-3                              (192,000)
│    │    └─ModuleList: 3-4                             632,709,120
│    │    └─LayerNorm: 3-5                              (2,560)
│    └─WhisperDecoder: 2-2                              --
│    │    └─Embedding: 3-6                              (66,388,480)
│    │    └─WhisperPositionalEmbedding: 3-7             (573,440)
│    │    └─ModuleList: 3-8                             (839,557,120)
│    │    └─LayerNorm: 3-9                              (2,560)
├─Sequential: 1-2                                       --
│    └─Conv1d: 2-3                 

In [7]:
decoder = AudioDecoder(rec_path, sample_rate=SAMPLE_RATE, num_channels=CHANNELS)
audio = decoder.get_all_samples().data.squeeze(0).to(device)

In [8]:
num_windows = int((audio.shape[0] - WINDOW * SAMPLE_RATE) // (STRIDE * SAMPLE_RATE) + 1)
num_windows

3637

In [9]:
windows = []
lengths = []
df = pd.DataFrame(columns=['start', 'end'])

for i in range(num_windows):
    start = int(i * STRIDE * SAMPLE_RATE)
    end = int(start + WINDOW * SAMPLE_RATE)
    segment = audio[start:end]
    windows.append(segment)
    lengths.append(torch.tensor(len(segment), device=device))
    df.loc[i] = [i * STRIDE, i * STRIDE + WINDOW]

windows = torch.stack(windows, dim=0)
lengths = torch.stack(lengths, dim=0)
windows.shape

torch.Size([3637, 32000])

In [10]:
fluency = []
disfluency_type = []

for i, j in slice_range(num_windows, 20):
    f, d = model.forward(windows[i:j], length=lengths[i:j])
    fluency.append(f.detach().cpu())
    disfluency_type.append(d.detach().cpu())

In [11]:
disfluency_labels = [
    'Block', 
    'Prolongation', 
    'Sound repetition', 
    'Word repetition', 
    'Interjection'
]

In [12]:
fluency_probs = F.softmax(torch.cat(fluency, dim=0), dim=1).numpy().astype(float)
df['fluency'] = fluency_probs[:, 0].round(3).tolist()
disfluency_preds = torch.sigmoid(torch.cat(disfluency_type, dim=0)).numpy()
disfluency_preds_mask = (disfluency_preds > DISFLUENCY_THRESHOLD).astype(int)
df['disfluency_type'] = [[label for pred, label in zip(preds, disfluency_labels) if pred == 1] for preds in disfluency_preds_mask]
df[:10]

,start,end,fluency,disfluency_type
0,0.0,2.0,0.717,[]
1,0.5,2.5,0.926,[]
2,1.0,3.0,0.971,[]
3,1.5,3.5,0.918,[]
4,2.0,4.0,0.406,[Interjection]
5,2.5,4.5,0.251,[Interjection]
6,3.0,5.0,0.332,[Interjection]
7,3.5,5.5,0.357,[Interjection]
8,4.0,6.0,0.870,[]
9,4.5,6.5,0.905,[]


In [13]:
df.to_parquet(out_path + '.parquet')
df.to_json(out_path + '.json', orient='records')